# 15 - accDM daughter: fluid-closure diagnostic + plateau ceff2 fit

1. **Diagnostic (Tasks 1-5):** does the effective sound speed collapse onto one function of `x=k/k_fs`? Pole-masked raw `delta_p/delta_rho`, `sigma/delta`. Universal at `x<1`, splits by **eta (not f)** at `x>1` -> `f<=0.3` is not the obstacle.
2. **Fit (Task 6):** the binned **median** `ceff2` is a **flat, k-independent plateau** `c_fs(eta)`, all **below 1/3** (the >1/3 excursions are oscillation peaks in the *envelope*, not the central value). So the closure is `ceff2(a;eta) = max(ca2(a), c_fs(eta))` - k-independent, always stable. The two-regime *step* was the wrong model (it railed x_t and underfit the plateau).
3. **Task 6b:** is `c_fs` just a background velocity-dispersion scale (so kappa/a_t enter for free)?
4. **Task 7:** does the plateau `c_fs` shift with `kappa` / `a_t`?

`w_sigma`/`w_theta` are zero in the exact hierarchy (`memory: w-sigma-zero-in-exact-hierarchy`). Published weight `W=1-2*eps_acc` *decreases* with eta while the data *increases* -> wrong sign (`memory: ceff2-fit-structure`). Decisive follow-up (Phase 3, post-rebuild): P(k) over **k<=1 only**.

In [ ]:
import sys; sys.path.insert(0, '.')          # import helpers from notebooks_test/
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from classy import Class
from fluid_closure_helpers import (
    ca2_from_kfs, mask_small_denom, log_upper_envelope, collapse_band,
)

plt.rcParams.update({
    'mathtext.fontset': 'stix', 'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'legend.fontsize': 9, 'lines.linewidth': 1.5, 'figure.dpi': 200})
qual_colors = ['#377eb8', '#ff7f00', '#4daf4a', '#f781bf', '#984ea3']

# --- base cosmology + accDM model (same setup as notebook 7) ---
omega_b, omega_cdm0 = 0.022383, 0.12011
A_s, n_s, tau_reio, H0 = 2.1005829616811546e-9, 0.96605, 0.0543, 67.32
base_params = {'omega_b': omega_b, 'omega_cdm': omega_cdm0, 'H0': H0,
               'A_s': A_s, 'n_s': n_s, 'tau_reio': tau_reio}
PREC = {'output': 'mPk', 'P_k_max_1/Mpc': 10.0, 'z_max_pk': 0.0,
        'evolver': 0, 'reionization_z_start_max': 80}

A_T, MASS, KAPPA = 0.13, 1e16, 6.0
A_REC = 1.0 / (1.0 + 1090.0)

# --- diagnostic sweep grids ---
K_GRID   = np.logspace(-2, 0.0, 18)          # 1/Mpc: sub-horizon, spans below to above k_fs
F_LIST   = [0.05, 0.1, 0.2, 0.3]             # daughter fraction up to the target
ETA_LIST = [0.1, 0.3, 1.0]                    # 3 boosts -> pin c_fs(eta)
DROP_FRAC = 0.2                               # pole mask: drop smallest 20% of |denominator|

def eps_acc_of_eta(eta):
    e2 = eta*eta
    return -e2 + np.sqrt(e2*e2 + 4*e2*eta + 5*e2 + 2*eta) - 2*eta

def accdm_params(eta, f_acc=0.1, kappa=KAPPA, a_t=A_T):
    """Exact-hierarchy accDM params for a given boost eta, fraction f_acc, and (kappa, a_t)."""
    ocdm = omega_cdm0 * (1 + f_acc*(1 - A_REC**kappa)/(1 + (A_REC/a_t)**kappa))**(-1)
    p = dict(base_params); p.update(PREC)
    p.update({'omega_cdm': ocdm,
              'vary_Gamma_acc': 'yes', 'kappa_acc': kappa, 'a_t_acc': a_t,
              'f_acc': f_acc, 'eta_acc': eta,
              'm_acc_in_GeV': MASS, 'm_cdm_in_GeV': MASS,
              'N_ncdm': 2, 'deg_ncdm': '3, 1',
              'm_ncdm': '0.02, {:.6e}'.format(MASS*1e9),
              'T_ncdm': '0.71611, 1', 'ncdm_quadrature_strategy': '0, 4',
              'ncdm_N_momentum_bins': '15, 501', 'N_ur': 0.00441,
              'background_Nloga': 5001, 'gauge': 'synchronous',
              'get_perturbations_in_current_gauge': 'yes',
              'ncdm_fluid_trigger_tau_over_tau_k': 25,
              'ncdm_fluid_approximation': 3})          # 3 = none (exact hierarchy)
    return p

print('setup OK; W(eta):', {e: round(1 - 2*eps_acc_of_eta(e), 3) for e in ETA_LIST},
      '<- decreases with eta (data increases -> W wrong sign)')

## Task 2 - tau-series extractor

In [ ]:
def _find_key(d, want):
    if want in d:
        return want
    for kk in d:
        if kk.replace(' ', '').startswith(want.replace(' ', '')):
            return kk
    raise KeyError('{!r} not found; available: {}'.format(want, list(d.keys())))

def extract_daughter_series(params, k_list):
    """Run exact CLASS; return {k: dict of daughter tau-series arrays}."""
    ks = np.sort(np.asarray(k_list, float))
    p = dict(params); p['k_output_values'] = ', '.join('{:.8e}'.format(k) for k in ks)
    M = Class(); M.set(p); M.compute()
    perts = M.get_perturbations()['scalar']
    bg = M.get_background()
    tau_bg = np.asarray(bg['conf. time [Mpc]'], float)
    a_bg   = 1.0 / (1.0 + np.asarray(bg['z'], float))
    H_bg   = np.asarray(bg['H [1/Mpc]'], float)
    o = np.argsort(tau_bg); tau_bg, a_bg, H_bg = tau_bg[o], a_bg[o], H_bg[o]
    kd  = _find_key(perts[0], 'delta_ncdm[1]'); kt  = _find_key(perts[0], 'theta_ncdm[1]')
    ksh = _find_key(perts[0], 'shear_ncdm[1]'); kc  = _find_key(perts[0], 'cs2_ncdm[1]')
    kf  = _find_key(perts[0], 'k_fss_acc[1]');  ktau = _find_key(perts[0], 'tau')
    out = {}
    for k, d in zip(ks, perts):
        tau = np.asarray(d[ktau], float)
        aH  = np.interp(tau, tau_bg, a_bg) * np.interp(tau, tau_bg, H_bg)
        out[k] = dict(tau=tau, aH=aH,
                      delta=np.asarray(d[kd], float), theta=np.asarray(d[kt], float),
                      shear=np.asarray(d[ksh], float), dpr=np.asarray(d[kc], float),
                      k_fs=np.asarray(d[kf], float))
    M.struct_cleanup(); M.empty()
    return out

In [ ]:
_probe = extract_daughter_series(accdm_params(ETA_LIST[0], f_acc=0.1), K_GRID[:3])
_s = _probe[K_GRID[0]]
for q in ('theta', 'shear', 'dpr', 'k_fs', 'aH', 'tau'):
    assert _s[q].shape == _s['delta'].shape, q
g = (_s['k_fs'] > 0) & np.isfinite(_s['aH'])
ca2 = ca2_from_kfs(_s['k_fs'][g], _s['aH'][g])
print('extractor OK; tau samples per k =', _s['delta'].size,
      '| ca2 in [{:.2e},{:.2e}] | x=k/k_fs reaches {:.0f}'.format(
          ca2.min(), ca2.max(), (K_GRID.max()/_s['k_fs'][g]).max()))
assert np.all(ca2 <= 1.0 + 1e-6), 'ca2 > 1 -> aH/k_fs mismatch (check background keys)'

## Task 3 - run the sweep once, cache series, build response envelopes

In [ ]:
def responses_from_series(ser, eta, drop_frac=DROP_FRAC):
    W = 1 - 2*eps_acc_of_eta(eta)
    Xc, Ce, Cf, Sd, Xv, Rv = [], [], [], [], [], []
    for k, s in ser.items():
        g = np.isfinite(s['k_fs']) & (s['k_fs'] > 0) & np.isfinite(s['aH'])
        if not np.any(g):
            continue
        x   = k / s['k_fs'][g]
        ca2 = ca2_from_kfs(s['k_fs'][g], s['aH'][g])
        delta = s['delta'][g]; theta = s['theta'][g]
        shear = s['shear'][g]; dpr = s['dpr'][g]
        safe_d = np.where(delta == 0, np.nan, delta)
        safe_t = np.where(theta == 0, np.nan, theta)
        Xc.append(x); Ce.append(np.abs(mask_small_denom(dpr, delta, drop_frac)))
        Cf.append(ca2 * (1 + 0.2*W*np.sqrt(x)))
        Sd.append(np.abs(mask_small_denom(shear/safe_d, delta, drop_frac)))
        Xv.append(x); Rv.append(np.abs(mask_small_denom(k*shear/safe_t, theta, drop_frac)))
    cat = np.concatenate
    return {'ceff2':   log_upper_envelope(cat(Xc), cat(Ce)),
            'ceff2_fit': log_upper_envelope(cat(Xc), cat(Cf)),
            'sig_del': log_upper_envelope(cat(Xc), cat(Sd)),
            'Rv':      log_upper_envelope(cat(Xv), cat(Rv))}

SERIES, RESPONSES = {}, {}
for eta in ETA_LIST:
    for f in tqdm(F_LIST, desc='eta={}'.format(eta)):
        ser = extract_daughter_series(accdm_params(eta, f_acc=f), K_GRID)
        SERIES[(f, eta)] = ser
        RESPONSES[(f, eta)] = responses_from_series(ser, eta)
print('cached', len(SERIES), 'series / responses')

## Task 4 - collapse plots + decision

In [ ]:
X_EVAL = np.logspace(-1, 3, 60)
ls_eta = {ETA_LIST[0]: '-', ETA_LIST[1]: '-.', ETA_LIST[2]: '--'}
col_f  = {f: qual_colors[i] for i, f in enumerate(F_LIST)}

def band_all(w):
    return collapse_band(X_EVAL, [RESPONSES[k][w] for k in RESPONSES])[1]
def band_fixed_eta(w):
    return float(np.nanmax([collapse_band(X_EVAL, [RESPONSES[(f, e)][w] for f in F_LIST])[1]
                            for e in ETA_LIST]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
for (f, eta), r in RESPONSES.items():
    for ax, w in zip(axes, ('ceff2', 'sig_del')):
        x, env = r[w]
        if x.size:
            ax.loglog(x, env, ls_eta[eta], color=col_f[f], alpha=0.8,
                      label='f={}, eta={}'.format(f, eta))
axes[0].axhline(1./3., color='red', ls='--', lw=1.0, label=r'causal $1/3$')
axes[0].set_title(r'raw $c_{\rm eff}^2=\delta p/\delta\rho$ envelope (pole-masked)')
axes[1].set_title(r'$\sigma/\delta$ envelope (pole-masked)')
for ax in axes:
    ax.set_xlabel(r'$x=k/k_{\rm fs}$'); ax.grid(True, which='both', alpha=0.3)
axes[0].legend(fontsize=6, ncol=2); plt.show()

band = {w: (band_all(w), band_fixed_eta(w)) for w in ('ceff2', 'sig_del')}
print('envelope band (pooled-all, fixed-eta):')
for w in ('ceff2', 'sig_del'):
    print('  {:>8}: {:.3f}, {:.3f}'.format(w, *band[w]))
print('(note: the envelope splits by eta; the MEDIAN plateau c_fs(eta) is fit in Task 6)')

## Task 5 - band vs x + kinematic cross-check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
bc, _ = collapse_band(X_EVAL, [RESPONSES[k]['ceff2']   for k in RESPONSES])
bs, _ = collapse_band(X_EVAL, [RESPONSES[k]['sig_del'] for k in RESPONSES])
axes[0].loglog(X_EVAL, bc, '-', color=qual_colors[0], label=r'$c_{\rm eff}^2$ band$(x)$')
axes[0].loglog(X_EVAL, bs, '-', color=qual_colors[2], label=r'$\sigma/\delta$ band$(x)$')
axes[0].axhline(0.15, color='red', ls='--', lw=1.0, label='0.15')
axes[0].axvline(1.0, color='gray', ls=':', lw=1.0, label=r'$x=1$')
axes[0].set_title('envelope band vs x'); axes[0].set_xlabel(r'$x=k/k_{\rm fs}$')
axes[0].set_ylabel('fractional band width'); axes[0].grid(True, which='both', alpha=0.3)
axes[0].legend(fontsize=8)
for (f, eta), r in RESPONSES.items():
    x, env = r['Rv']
    if x.size:
        axes[1].loglog(x, env, ls_eta[eta], color=col_f[f], alpha=0.8)
axes[1].set_title(r'cross-check $R_v=k\sigma/\theta$ (pole-masked)')
axes[1].set_xlabel(r'$x=k/k_{\rm fs}$'); axes[1].grid(True, which='both', alpha=0.3)
plt.show()

## Task 6 - plateau fit `ceff2(a; eta) = max(ca2(a), c_fs(eta))`

The binned **median** `ceff2` is flat in `x`, so `c_fs(eta)` = the median pole-masked `ceff2` over the plateau window `x in [0.3, 500]` (pooled over `f`). No k-dependence, no railing. Compared against the daughter data and the old `sqrt/W` form.

In [ ]:
def binned_samples(eta, n_bins=30, min_per_bin=4):
    """Pool pole-masked (x, ca2, ceff2) over f at this eta; robust median per log-x bin."""
    xs, cas, ces = [], [], []
    for f in F_LIST:
        for k, s in SERIES[(f, eta)].items():
            g = np.isfinite(s['k_fs']) & (s['k_fs'] > 0) & np.isfinite(s['aH'])
            if not np.any(g):
                continue
            x = k / s['k_fs'][g]
            ca2 = ca2_from_kfs(s['k_fs'][g], s['aH'][g])
            ce = np.abs(mask_small_denom(s['dpr'][g], s['delta'][g], DROP_FRAC))
            m = np.isfinite(ce) & (ce > 0)
            xs.append(x[m]); cas.append(ca2[m]); ces.append(ce[m])
    x = np.concatenate(xs); ca2 = np.concatenate(cas); ce = np.concatenate(ces)
    edges = np.logspace(np.log10(x.min()), np.log10(x.max()), n_bins + 1)
    idx = np.clip(np.digitize(x, edges) - 1, 0, n_bins - 1)
    xb, cab, deb = [], [], []
    for b in range(n_bins):
        sel = idx == b
        if sel.sum() >= min_per_bin:
            xb.append(np.sqrt(edges[b]*edges[b+1]))
            cab.append(np.median(ca2[sel])); deb.append(np.median(ce[sel]))
    return np.array(xb), np.array(cab), np.array(deb)

def plateau_cfs(eta, x_lo=0.3, x_hi=500.0):
    """c_fs(eta) = median pole-masked ceff2 over the flat plateau window (pooled over f)."""
    vals = []
    for f in F_LIST:
        for k, s in SERIES[(f, eta)].items():
            g = np.isfinite(s['k_fs']) & (s['k_fs'] > 0) & np.isfinite(s['aH'])
            if not np.any(g):
                continue
            x = k / s['k_fs'][g]
            ce = np.abs(mask_small_denom(s['dpr'][g], s['delta'][g], DROP_FRAC))
            m = np.isfinite(ce) & (ce > 0) & (x >= x_lo) & (x <= x_hi)
            if np.any(m):
                vals.append(ce[m])
    return float(np.median(np.concatenate(vals)))

BIN = {eta: binned_samples(eta) for eta in ETA_LIST}
CFS = {eta: plateau_cfs(eta) for eta in ETA_LIST}
print('plateau c_fs(eta) =', {e: round(CFS[e], 4) for e in ETA_LIST})
print('all below causal 1/3:', all(CFS[e] < 1./3. for e in ETA_LIST),
      '| monotone in eta:', bool(np.all(np.diff([CFS[e] for e in ETA_LIST]) > 0)))

In [ ]:
def model_plateau(ca2, eta):
    return np.maximum(np.asarray(ca2, float), CFS[eta])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
for eta, c in zip(ETA_LIST, qual_colors):
    xb, cab, deb = BIN[eta]
    axes[0].loglog(xb, deb, 'o', color=c, ms=4, label='data eta={}'.format(eta))
    axes[0].loglog(xb, model_plateau(cab, eta), '-', color=c)
    W = 1 - 2*eps_acc_of_eta(eta)
    axes[0].loglog(xb, np.abs(cab*(1 + 0.2*W*np.sqrt(xb))), ':', color=c, alpha=0.6)
axes[0].axhline(1./3., color='red', ls='--', lw=1.0)
axes[0].set_title('plateau max(ca2,c_fs) (-) vs data (o) vs old sqrt/W (:)')
axes[0].set_xlabel(r'$x=k/k_{\rm fs}$'); axes[0].set_ylabel(r'$c_{\rm eff}^2$')
axes[0].grid(True, which='both', alpha=0.3); axes[0].legend(fontsize=7)
etas = np.array(ETA_LIST); cvals = np.array([CFS[e] for e in ETA_LIST])
axes[1].plot(etas, cvals, 'o-', color=qual_colors[0])
axes[1].axhline(1./3., color='red', ls='--', lw=1.0, label='1/3')
axes[1].set_title(r'$c_{\rm fs}(\eta)$ (all < 1/3)')
axes[1].set_xlabel(r'$\eta$'); axes[1].set_ylabel(r'$c_{\rm fs}$')
axes[1].grid(alpha=0.3); axes[1].legend()
plt.show()

print('plateau log-RMS residual (x in [0.3,500], lower is better):')
for eta in ETA_LIST:
    xb, cab, deb = BIN[eta]; W = 1 - 2*eps_acc_of_eta(eta)
    m = (xb >= 0.3) & (xb <= 500)
    r_new = np.sqrt(np.mean((np.log(model_plateau(cab[m], eta)) - np.log(deb[m]))**2))
    r_old = np.sqrt(np.mean((np.log(np.abs(cab[m]*(1 + 0.2*W*np.sqrt(xb[m])))) - np.log(deb[m]))**2))
    print('  eta={}: plateau {:.3f}  vs  old sqrt/W {:.3f}'.format(eta, r_new, r_old))

## Task 6b - is `c_fs` a background velocity-dispersion scale?

If `c_fs(eta)` is a fixed multiple of a background dispersion proxy (here the peak `ca2` over the trajectory), then it is not a free knob - it carries `eta`, `a_t`, `kappa` automatically through the background. Look for a ~constant `c_fs / peak_ca2` ratio across eta.

In [ ]:
peak_ca2 = {}
for eta in ETA_LIST:
    vals = []
    for f in F_LIST:
        for k, s in SERIES[(f, eta)].items():
            g = (s['k_fs'] > 0) & np.isfinite(s['aH'])
            if np.any(g):
                vals.append(ca2_from_kfs(s['k_fs'][g], s['aH'][g]))
    peak_ca2[eta] = float(np.nanmax(np.concatenate(vals)))

print('eta    c_fs    peak_ca2   c_fs/peak_ca2')
for eta in ETA_LIST:
    print('{:>4}   {:.4f}   {:.4f}    {:.2f}'.format(
        eta, CFS[eta], peak_ca2[eta], CFS[eta]/peak_ca2[eta]))
print('-> ~constant ratio => c_fs is a background dispersion scale (kappa/a_t enter for free).')

plt.figure(figsize=(5, 4))
plt.loglog([peak_ca2[e] for e in ETA_LIST], [CFS[e] for e in ETA_LIST], 'o-', color=qual_colors[0])
for e in ETA_LIST:
    plt.annotate('eta={}'.format(e), (peak_ca2[e], CFS[e]), fontsize=8)
plt.xlabel(r'peak $c_a^2$ (background dispersion proxy)'); plt.ylabel(r'$c_{\rm fs}$')
plt.title(r'$c_{\rm fs}$ vs background dispersion'); plt.grid(True, which='both', alpha=0.3); plt.show()

## Task 7 - sensitivity of the plateau `c_fs` to `kappa` and `a_t`

Fix `eta=0.1, f=0.1`; vary `kappa` alone and `a_t` alone. If the `ceff2(x)` envelopes and the plateau `c_fs` barely move, the `k_fs` normalization absorbs `kappa`/`a_t` and `c_fs(eta)` stands; if they split, `c_fs` also depends on the `f(q)` shape.

In [ ]:
ETA_S, F_S = 0.1, 0.1
KAPPA_SWEEP = [2.0, 6.0, 20.0]      # a_t = 0.13 fixed
AT_SWEEP    = [0.01, 0.13, 0.5]     # kappa = 6.0 fixed

def cfs_and_env(kappa, a_t, x_lo=0.3, x_hi=500.0):
    ser = extract_daughter_series(accdm_params(ETA_S, f_acc=F_S, kappa=kappa, a_t=a_t), K_GRID)
    Xc, Ce, plat = [], [], []
    for k, s in ser.items():
        g = np.isfinite(s['k_fs']) & (s['k_fs'] > 0) & np.isfinite(s['aH'])
        if not np.any(g):
            continue
        x = k / s['k_fs'][g]
        ce = np.abs(mask_small_denom(s['dpr'][g], s['delta'][g], DROP_FRAC))
        Xc.append(x); Ce.append(ce)
        m = np.isfinite(ce) & (ce > 0) & (x >= x_lo) & (x <= x_hi)
        if np.any(m):
            plat.append(ce[m])
    env = log_upper_envelope(np.concatenate(Xc), np.concatenate(Ce))
    return env, float(np.median(np.concatenate(plat)))

SENS, CFS_S, base = {}, {}, None
for kap in tqdm(KAPPA_SWEEP, desc='kappa'):
    SENS[('kappa', kap)], CFS_S[('kappa', kap)] = cfs_and_env(kap, 0.13)
    if kap == 6.0:
        base = (SENS[('kappa', kap)], CFS_S[('kappa', kap)])
for at in tqdm(AT_SWEEP, desc='a_t'):
    if at == 0.13:
        SENS[('a_t', at)], CFS_S[('a_t', at)] = base
    else:
        SENS[('a_t', at)], CFS_S[('a_t', at)] = cfs_and_env(6.0, at)
print('c_fs across kappa {2,6,20}:  ', [round(CFS_S[('kappa', k)], 4) for k in KAPPA_SWEEP])
print('c_fs across a_t {0.01,.13,.5}:', [round(CFS_S[('a_t', a)], 4) for a in AT_SWEEP])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
for kap, c in zip(KAPPA_SWEEP, qual_colors):
    x, env = SENS[('kappa', kap)]
    axes[0].loglog(x, env, '-', color=c, label='kappa={} (c_fs={:.3f})'.format(kap, CFS_S[('kappa', kap)]))
for at, c in zip(AT_SWEEP, qual_colors):
    x, env = SENS[('a_t', at)]
    axes[1].loglog(x, env, '-', color=c, label='a_t={} (c_fs={:.3f})'.format(at, CFS_S[('a_t', at)]))
for ax, t in zip(axes, ('vary kappa (a_t=0.13)', 'vary a_t (kappa=6)')):
    ax.axhline(1./3., color='red', ls='--', lw=1.0); ax.axvline(1.0, color='gray', ls=':', lw=1.0)
    ax.set_xlabel(r'$x=k/k_{\rm fs}$'); ax.set_title(t)
    ax.grid(True, which='both', alpha=0.3); ax.legend(fontsize=8)
axes[0].set_ylabel(r'$c_{\rm eff}^2$'); plt.show()

b_kappa = collapse_band(X_EVAL, [SENS[('kappa', k)] for k in KAPPA_SWEEP])[1]
b_at    = collapse_band(X_EVAL, [SENS[('a_t', a)] for a in AT_SWEEP])[1]
ck = [CFS_S[('kappa', k)] for k in KAPPA_SWEEP]; caa = [CFS_S[('a_t', a)] for a in AT_SWEEP]
print('envelope band  across kappa: {:.3f} | across a_t: {:.3f}'.format(b_kappa, b_at))
print('c_fs spread    across kappa: {:.1%} | across a_t: {:.1%}'.format(
    (max(ck)-min(ck))/np.median(ck), (max(caa)-min(caa))/np.median(caa)))
print('-> small: x-normalization absorbs kappa/a_t; c_fs(eta) stands.', 'large: c_fs depends on them (shape effect).')

## Verdict + next step

*(Fill from the printed output.)*

- **Closure form:** median `ceff2` is a flat k-independent plateau -> `ceff2(a;eta) = max(ca2(a), c_fs(eta))`. `c_fs = {0.1: [FILL], 0.3: [FILL], 1.0: [FILL]}`, all **< 1/3** -> naturally stable. Plateau log-RMS [FILL] vs old sqrt/W [FILL].
- **c_fs origin (6b):** `c_fs/peak_ca2` = [FILL] across eta -> `c_fs` [is / is not] a background dispersion scale.
- **kappa/a_t sensitivity (7):** c_fs spread across kappa = [FILL], across a_t = [FILL]. -> `c_fs` [is absorbed by x-normalization / also depends on kappa,a_t].
- **Phase 2 (C, rebuild):** `ncdm_ceff2_mode = 2` = `max(ca2, c_fs(eta[,a_t,kappa]))` in `perturbations_ceff2_ncdm` (no cap needed if c_fs<1/3; keep the 1/3 cap as a guard).
- **Phase 3 (decisive):** fluid mode-2 vs exact, **P(k) residual over k<=1 only**, at `f=0.3, eta=0.1`. Success = ~1%. Since the plateau is sub-1/3 and k-independent, the nb7 high-k blow-up should not recur at k<=1.

**Keep in sync:** `fluid_closure_helpers.py` (unit-tested). If Phase 2 uses `max(ca2, c_fs)`, no new helper shape is needed - `c_fs(eta[,a_t,kappa])` is a scalar law from this notebook.